In [ ]:
# Configuracao do ambiente (celula de setup do notebook)
import sys, os, warnings
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', 'src')))
warnings.filterwarnings('ignore')
%matplotlib inline

# Capítulo 25 — DEPLOYMENT: DO NOTEBOOK PARA O MUNDO REAL

> "Um modelo que só existe num notebook é uma filosofia. Um modelo em produção é uma ferramenta. A filosofia pode ser bela, mas só a ferramenta muda o mundo."
> — Um Engenheiro de MLOps

Meu trabalho vivia em notebooks — artefatos de pesquisa, um diário da jornada. Mas um sistema de diagnóstico não tem valor num laboratório: seu valor está na clínica, na mão do médico. Eu precisava tirar o modelo do conforto do notebook e expô-lo ao mundo real.

Isso significava empacotar o ensemble num objeto serializável e criar uma interface para que outros sistemas o consultassem — enviando os dados de um paciente e recebendo uma previsão. A forma mais comum: uma **API**.

## Do Modelo ao Serviço

O fluxo de *deployment* tem quatro passos:

**1. Treinar o modelo final** em **todos** os dados disponíveis (treino + teste). A avaliação honesta já foi feita — a estimativa (Capítulos 15, 21) e o veredito final no cofre (Capítulo 24); agora o objetivo é o artefato mais robusto possível.

**2. Serializar** o modelo em arquivo, com `joblib`, para recarregá-lo sem retreinar.

**3. Expor via API** (usaremos o **FastAPI**), com um *endpoint* `/prever` que recebe os dados de um paciente em JSON e devolve a previsão.

**4. (Opcional) Conteinerizar** com Docker para portabilidade.

#### Código 25.1: Treinar e Salvar o Modelo Final


In [ ]:
# treinar_modelo.py
import joblib
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier, VotingClassifier
from oncoclassify_utils import X, y   # base canonica COMPLETA (600 amostras)

log = Pipeline([("sc", StandardScaler()), ("lr", LogisticRegression(solver="liblinear", random_state=42))])
svm = Pipeline([("sc", StandardScaler()), ("svm", SVC(kernel="rbf", C=10, gamma=0.01, probability=True, random_state=42))])
gb  = GradientBoostingClassifier(n_estimators=100, random_state=42)

modelo_final = VotingClassifier([("lr", log), ("svm", svm), ("gb", gb)], voting="soft")
modelo_final.fit(X, y)   # treina em TODAS as 600 amostras

joblib.dump(modelo_final, "oncoclassify_v2.pkl")
print(f"Modelo treinado em {len(X)} amostras e salvo em 'oncoclassify_v2.pkl'")

Modelo treinado em 600 amostras e salvo em 'oncoclassify_v2.pkl'


#### Código 25.2: A API com FastAPI


In [ ]:
# app.py  -- rodar com: uvicorn app:app --reload
import joblib
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel

modelo = joblib.load("oncoclassify_v2.pkl")
# os nomes/ordem das features vêm do proprio modelo -> robustez
FEATURES = list(modelo.feature_names_in_)

app = FastAPI(title="OncoClassify API", version="2.0")

# Recebemos um dicionario {feature: valor}, nao uma lista posicional.
# Assim a ordem das features nao importa -- evita erros silenciosos.
class Paciente(BaseModel):
    features: dict[str, float]

@app.post("/prever")
def prever(dados: Paciente):
    try:
        entrada = pd.DataFrame([dados.features])[FEATURES]  # reordena pela ordem do modelo
        prob = modelo.predict_proba(entrada)[0]
        return {
            "diagnostico": "Maligno" if prob[0] >= 0.5 else "Benigno",
            "probabilidade_maligno": round(float(prob[0]), 4),
            "probabilidade_benigno": round(float(prob[1]), 4),
        }
    except KeyError as e:
        return {"erro": f"feature ausente: {e}"}

Repare na escolha de projeto: a API recebe um **dicionário nomeado** `{feature: valor}`, não uma lista posicional. Reordenamos as colunas pela ordem que o próprio modelo guarda (`feature_names_in_`). Assim, se o sistema cliente enviar as *features* em outra ordem, nada quebra silenciosamente — e uma *feature* ausente gera um erro claro, não uma previsão errada. É a mesma disciplina anti-erro dos *pipelines*, agora na fronteira do sistema.

Para rodar: `python treinar_modelo.py` e depois `uvicorn app:app --reload`. O FastAPI gera documentação interativa em `http://127.0.0.1:8000/docs`, onde o *endpoint* `/prever` pode ser testado.

> **📌 CHECKLIST DO CAPÍTULO 25**
>
> ✅ Entende o que é *deployment* e por que é necessário.
>
> ✅ Sabe os passos: treinar o modelo final em todos os dados, serializar com `joblib`, expor via API.
>
> ✅ Executou o Código 25.1 e salvou o modelo final.
>
> ✅ Entende por que uma API que recebe *features* **nomeadas** é mais robusta que uma lista posicional.
>
> **⚠️ CRÍTICO:** Ao treinar o modelo para produção, é prática padrão usar todos os dados disponíveis — você já tem uma estimativa de desempenho confiável da fase de avaliação.

Ver a documentação interativa da API funcionando foi um momento de profunda satisfação. O OncoClassify 2.0 não era mais um conceito abstrato: era um serviço com um endereço na web, capaz de responder. Eu imaginava um prontuário eletrônico consultando `/prever` e recebendo de volta uma probabilidade que ajudaria um médico.

Mas colocar o modelo em produção era só o começo. O mundo real é caótico e mutável. Os dados podem mudar; o desempenho, que era excelente hoje, pode se degradar em silêncio. Um modelo em produção é um sistema vivo, que precisa ser observado. Eu precisava de um plano de **monitoramento**.
